In [1]:
from datasets import load_dataset, get_dataset_split_names
import  pandas as pd

In [2]:
emotion = load_dataset(path="dair-ai/emotion")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [3]:
emotion.set_format(type="pandas")
emotion


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})

In [4]:
df = emotion['train'][:]
df.head(), df.shape

(                                                text  label
 0                            i didnt feel humiliated      0
 1  i can go from feeling so hopeless to so damned...      0
 2   im grabbing a minute to post i feel greedy wrong      3
 3  i am ever feeling nostalgic about the fireplac...      2
 4                               i am feeling grouchy      3,
 (16000, 2))

In [5]:
emotion['train'].features['label'].names

['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']

In [6]:
classes = emotion['train'].features['label'].names
classes

['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']

In [7]:
df['label_name'] = df['label'].map(lambda x: classes[x])
df.head()

,text,label,label_name
0,i didnt feel humiliated,0,sadness
1,i can go from feeling so hopeless to so damned...,0,sadness
2,im grabbing a minute to post i feel greedy wrong,3,anger
3,i am ever feeling nostalgic about the fireplac...,2,love
4,i am feeling grouchy,3,anger


In [8]:
label_counts = df['label_name'].value_counts()
(label_counts)

,count
label_name,
joy,5362
sadness,4666
anger,2159
fear,1937
love,1304
surprise,572


## Data Analysis

In [9]:
import matplotlib.pyplot as plt
import plotly.express as px


In [10]:
fig = px.bar(label_counts, title='Distribution of Emotion Labels')
fig.show()

In [11]:
df['words_per_tweet'] = df['text'].str.split().apply(len)
df['words_per_tweet']

,words_per_tweet
0,4
1,21
2,10
3,18
4,4
...,...
15995,24
15996,20
15997,6
15998,14


Box plot for max tokens

In [12]:
fig = px.box(df, y="words_per_tweet", x='label_name')
fig.show()

## Text to Token conversion

In [13]:
from transformers import AutoTokenizer
model_ckpt = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [14]:
text='i love machine learning. Fine-Tuning DistilBERT for Emotion Recognition using HuggingFace'

enc_text = tokenizer(text)
enc_text

{'input_ids': [101, 1045, 2293, 3698, 4083, 1012, 2986, 1011, 17372, 4487, 16643, 23373, 2005, 7603, 5038, 2478, 17662, 12172, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [15]:
tokenizer.convert_ids_to_tokens(enc_text.input_ids)

['[CLS]',
 'i',
 'love',
 'machine',
 'learning',
 '.',
 'fine',
 '-',
 'tuning',
 'di',
 '##sti',
 '##lbert',
 'for',
 'emotion',
 'recognition',
 'using',
 'hugging',
 '##face',
 '[SEP]']

In [16]:
tokenizer.vocab_size, tokenizer.model_max_length

(30522, 512)

### Tokenization of the Emotion Data

In [17]:
emotion.reset_format()

In [18]:
def tokenize(batch):
  temp = tokenizer(batch['text'], padding=True, truncation=True, max_length=512)
  return temp

In [19]:
emotion['train'][:1]

{'text': ['i didnt feel humiliated'], 'label': [0]}

In [20]:
tokenize(emotion['train'][:1])

{'input_ids': [[101, 1045, 2134, 2102, 2514, 26608, 102]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1]]}

In [21]:
emotions_enc = emotion.map(tokenize, batched=True, batch_size=None)

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [22]:
emotions_enc


DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2000
    })
})

In [23]:
inputs = tokenizer(text, return_tensors='pt')

In [24]:
from transformers import AutoModel,utils
import torch

In [25]:
model = AutoModel.from_pretrained(model_ckpt,output_attentions=True)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [26]:
model

DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSelfAttention(
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): L

In [27]:
with torch.no_grad():
  outputs = model(**inputs)

last_hidden_states = outputs.last_hidden_state
last_hidden_states

tensor([[[-0.1818, -0.0233, -0.1346,  ..., -0.0864,  0.2466,  0.3884],
         [ 0.4097,  0.2444, -0.1326,  ...,  0.0277,  0.4845,  0.3026],
         [ 0.6660,  0.6691,  0.5440,  ..., -0.0618,  0.3328,  0.0826],
         ...,
         [-0.1301, -0.0351,  0.5981,  ..., -0.1502, -0.0932,  0.2471],
         [ 0.0312,  0.0320,  0.1079,  ..., -0.0120, -0.2340,  0.1610],
         [ 0.8394,  0.4524, -0.4639,  ...,  0.0585, -0.7900, -0.1886]]])

## Fine tuning for classification

In [28]:
from transformers import AutoModelForSequenceClassification

In [29]:
num_labels = len(classes)

In [30]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
device

device(type='cuda')

In [31]:
model = AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=num_labels).to(device)
model

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [33]:
from transformers import TrainingArguments

batch_size = 64

model_name = 'distilbert-finetuned-emotion'

training_args = TrainingArguments(output_dir=model_name,
                                  num_train_epochs=2,
                                  per_device_train_batch_size=batch_size,
                                  per_device_eval_batch_size=batch_size,
                                  eval_strategy='epoch',
                                  weight_decay=0.01,
                                  learning_rate=2e-5
                                  )

In [34]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(pred):
  labels = pred.label_ids
  preds = pred.predictions.argmax(-1)
  f1 = f1_score(labels, preds, average='weighted')
  acc = accuracy_score(labels, preds)
  return {'accuracy': acc, 'f1': f1}

In [40]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset = emotions_enc['train'],
    eval_dataset=emotions_enc['validation'],
    processing_class=tokenizer
    )

In [41]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.259605,0.916000,0.915957
2,0.440227,0.200507,0.928500,0.928507


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=500, training_loss=0.44022708129882815, metrics={'train_runtime': 260.5144, 'train_samples_per_second': 122.834, 'train_steps_per_second': 1.919, 'total_flos': 720342861696000.0, 'train_loss': 0.44022708129882815, 'epoch': 2.0})

In [48]:
pred_outputs = trainer.predict(emotions_enc['test'])
print(pred_outputs.metrics)
print(pred_outputs.predictions)

{'test_loss': 0.20681911706924438, 'test_accuracy': 0.9155, 'test_f1': 0.9154145103453254, 'test_runtime': 4.3159, 'test_samples_per_second': 463.404, 'test_steps_per_second': 7.414}
[[ 4.6893167  -0.8318769  -1.6231083  -0.24732167 -1.0952526  -1.6815635 ]
 [ 4.7658906  -0.9553813  -1.499374   -0.59318596 -1.0532293  -1.5226915 ]
 [ 4.688561   -1.0626513  -1.4649616  -0.9205231  -0.7622896  -1.5108267 ]
 ...
 [-0.977869    4.950732   -0.51326334 -1.3737049  -1.4962837  -1.0407826 ]
 [-0.9123739   4.772118   -0.5120981  -1.6874572  -1.0681906  -1.1518803 ]
 [-1.283695   -1.227074   -1.4818505  -1.1661533   2.4580944   2.0639474 ]]


In [43]:
import numpy as np

y_preds = np.argmax(pred_outputs.predictions, axis=1)
y_true = emotions_enc['test'][:]['label']


In [47]:
from sklearn.metrics import classification_report

print(classification_report(y_true, y_preds, target_names=classes))


              precision    recall  f1-score   support

     sadness       0.95      0.96      0.96       581
         joy       0.93      0.94      0.93       695
        love       0.77      0.82      0.80       159
       anger       0.93      0.91      0.92       275
        fear       0.88      0.88      0.88       224
    surprise       0.78      0.70      0.74        66

    accuracy                           0.92      2000
   macro avg       0.88      0.87      0.87      2000
weighted avg       0.92      0.92      0.92      2000



In [49]:
!zip -r distilbert-finetuned-emotion.zip distilbert-finetuned-emotion


  adding: distilbert-finetuned-emotion/ (stored 0%)
  adding: distilbert-finetuned-emotion/checkpoint-500/ (stored 0%)
  adding: distilbert-finetuned-emotion/checkpoint-500/training_args.bin (deflated 53%)
  adding: distilbert-finetuned-emotion/checkpoint-500/trainer_state.json (deflated 57%)
  adding: distilbert-finetuned-emotion/checkpoint-500/config.json (deflated 55%)
  adding: distilbert-finetuned-emotion/checkpoint-500/optimizer.pt (deflated 27%)
  adding: distilbert-finetuned-emotion/checkpoint-500/tokenizer_config.json (deflated 42%)
  adding: distilbert-finetuned-emotion/checkpoint-500/rng_state.pth (deflated 26%)
  adding: distilbert-finetuned-emotion/checkpoint-500/scheduler.pt (deflated 62%)
  adding: distilbert-finetuned-emotion/checkpoint-500/tokenizer.json (deflated 71%)
  adding: distilbert-finetuned-emotion/checkpoint-500/model.safetensors (deflated 8%)
